In [1]:
import numpy as np
import pandas as pd
import os
from matplotlib import pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import optuna
import shap
import pickle
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
class DataProcessor:
    def __init__(self, all_data, desc_set, object_variable, split_seed):
        self.desc_set = desc_set
        self.object_variable = object_variable
        self.split_seed = split_seed
        self.X_data, self.y_data ,self.X_train, self.X_test, self.y_train, self.y_test = self.prepare_datasets(all_data)
        self.Y_data, self.Y_train, self.Y_test = self.normalize_data()
        self.dall, self.dtrain, self.dtest, self.evals = self.convert_data()

    def prepare_datasets(self, all_data):
        X_data = all_data[self.desc_set]
        y_data = pd.DataFrame(all_data[self.object_variable])
        
        train, test = train_test_split(all_data, test_size=0.2, random_state=self.split_seed)
        X_train = train[self.desc_set]
        y_train = pd.DataFrame(train[self.object_variable])
        X_test = test[self.desc_set]
        y_test = pd.DataFrame(test[self.object_variable])

        return X_data, y_data, X_train, X_test, y_train, y_test

    def normalize_data(self):
        self.mean_data_y = self.y_data.mean()
        self.std_data_y = self.y_data.std()
        Y_data = (self.y_data - self.mean_data_y) / self.std_data_y
        
        self.mean_train_y = self.y_train.mean()
        self.std_train_y = self.y_train.std()
        self.mean_test_y = self.y_test.mean()
        self.std_test_y = self.y_test.std()
        Y_train = (self.y_train - self.mean_train_y) / self.std_train_y
        Y_test = (self.y_test - self.mean_test_y) / self.std_test_y
        
        return  Y_data, Y_train, Y_test
    
    def convert_data(self):
        dall = xgb.DMatrix(self.X_data, label=self.Y_data)
        dtrain = xgb.DMatrix(self.X_train, label=self.Y_train)
        dtest = xgb.DMatrix(self.X_test, label=self.Y_test)
        evals = [(dtest, 'eval'), (dtrain, 'train')]
        
        return dall, dtrain, dtest, evals

In [3]:
class Optimization:
    def __init__(self, data_processor, directory, basename, n_trials=300):
        self.dp = data_processor
        self.directory = directory
        self.basename = basename
        self.n_trials = n_trials
 
    def objective(self, trial):
        
        params = {
            "verbosity": 0,
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "booster":  "gbtree",
            'n_estimators' : 10000,
            'base_score' : float(self.dp.Y_data.mean().values),
            'max_depth': trial.suggest_int('max_depth', 2, 5),
            'max_leaves': trial.suggest_int('max_leaves', 2, 5),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.1, 1),
            'colsample_byrevel': trial.suggest_float('colsample_bylevel', 0.1, 1),
            'colsample_bybode': trial.suggest_float('colsample_bynode', 0.1, 1),
            'subsample': trial.suggest_float('subsample', 0, 1),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
            "leraning_rate" : trial.suggest_float("leraning_rate", 1e-4, 10, log=False),
            "gamma" : trial.suggest_float("gamma", 0.1, 10, log=False),
        }

        cv_results = xgb.cv(
            params = params,
            dtrain = self.dp.dall,
            num_boost_round=10000,
            seed=0,
            nfold=5, # CVの分割数
            metrics={"rmse"},
            early_stopping_rounds=300,
            shuffle=True,
        )
        
        return cv_results["test-rmse-mean"].mean()

    def optimize_parameters(self):
        max_depth = 5
        max_leaves = 4
        colsample_bytree = 0.3402744077431494
        colsample_bylevel = 0.8392492581311292
        colsample_bynode = 0.7347853808112024
        subsample = 0.8515695846938984
        reg_lambda = 0.0003194232244540776
        reg_alpha = 0.41955205249976124
        leraning_rate = 8.078050956559998
        gamma = 0.13872463795888207

        
        start_params = {
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',
            "booster":  "gbtree",
            'verbosity' : 0,
            'random_state' : 0,
            'n_estimators' : 10000,
            'base_score' : float(self.dp.Y_data.mean().values), 
            'max_depth': max_depth, 
            'max_leaves':max_leaves,
            'colsample_bytree': colsample_bytree,
            'colsample_byrevel': colsample_bylevel,
            'colsample_bynode': colsample_bynode,
            'subsample': subsample,
            "reg_lambda": reg_lambda,
            "reg_alpha": reg_alpha,
            "leraning_rate" : leraning_rate,
            "gamma" : gamma,
        }
        self.start_params = start_params
        # ベイズ最適化実行
        study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
        study.enqueue_trial(self.start_params)
        study.optimize(lambda trial: self.objective(trial), n_trials=self.n_trials)
        self.opt_params = study.best_params
        
        with open(f"{self.directory}/{self.basename}_best_params", 'w') as f:
            for k, v in self.opt_params.items():
                print(f'{k} = {v}', file=f)

In [4]:
class Modeling:
    def __init__(self, data_processor, opt, directory, basename, n_seed=100):
        self.dp = data_processor
        self.opt = opt
        self.directory = directory
        self.basename = basename
        self.n_seed = n_seed
    
    def search_best_model(self, i):
        
        #複数のseed値で探索
        best_score = 0
        best_seed = 0
        best_params = {}
        for seed in range(self.n_seed):
            base_params = {"verbosity": 0,
                "objective": "reg:squarederror",
                "eval_metric": "rmse",
                "booster":  "gbtree",
                'n_estimators' : 10000,
                'random_state' : int(seed)
                }
            best_params = base_params
            all_params = dict(**base_params, **self.opt.opt_params) 
            model = xgb.train(params=all_params,
                              dtrain=self.dp.dtrain,
                              num_boost_round=10000,
                              early_stopping_rounds=300, 
                              evals=self.dp.evals, 
                              verbose_eval=False
                             )
            
            #Predict
            train_pred = pd.DataFrame(model.predict(self.dp.dtrain), columns=[self.dp.object_variable])
            test_pred = pd.DataFrame(model.predict(self.dp.dtest), columns=[self.dp.object_variable])
            # R2値を表示
            r2_train = r2_score(self.dp.Y_train, train_pred).round(4)
            r2_test = r2_score(self.dp.Y_test, test_pred).round(4)
            
            if r2_test >= best_score:
                best_score = r2_test
                best_seed = seed
                best_params = all_params
            print(f'Spliting Data {i} : seed {seed} done, best_seed = {best_seed}, best_score = {best_score}')

        self.best_params = best_params
        with open(f"{self.directory}/{self.basename}_"+str(i), 'w') as f:
            for k, v in self.best_params.items():
                print(f'{k} = {v}', file=f)

        return best_params

In [5]:
class ModelSaver:
    def __init__(self, modeler, data_processor, directory, basename):
        self.dp = data_processor
        self.model = modeler
        self.directory = directory
        self.basename = basename

    def save_best_model(self, i):
        best_model = xgb.train(params=self.model.best_params,
                           dtrain=self.dp.dtrain,
                           num_boost_round=10000,
                           early_stopping_rounds=100,
                           evals=self.dp.evals, 
                           verbose_eval=False,
                          )
        
        with open(f"{self.directory}/{self.basename}_model_{i}.pickle", mode='wb') as f:
            pickle.dump(best_model, f)
        best_model.save_model(f"{self.directory}/{self.basename}_model_{i}.json")
        
        train_pred = pd.DataFrame(best_model.predict(self.dp.dtrain), columns=['pIC50'])
        test_pred = pd.DataFrame(best_model.predict(self.dp.dtest), columns=['pIC50'])
        r2_train = r2_score(self.dp.Y_train, train_pred).round(4)
        r2_test = r2_score(self.dp.Y_test, test_pred).round(4)
        self.train_pred = train_pred
        self.test_pred = test_pred
        self.r2_train = r2_train
        self.r2_test = r2_test
        self.best_model = best_model
        
        
        return train_pred, test_pred, r2_train, r2_test

    def save_yyplot(self, i):
        m1, s1 = self.dp.mean_train_y, self.dp.std_train_y
        m2, s2 = self.dp.mean_test_y, self.dp.std_test_y
        # 説明変数と目的変数のデータ点の散布図をプロット
        fig = plt.figure(figsize=(5,5))
        fig.suptitle(" ", fontsize=15)
        s=np.linspace(2.5, 7.5, 50)
        t = s
        plt.plot(s, t, color='k')
        #scatter
        plt.scatter(m1 + self.train_pred*s1, m1 + self.dp.Y_train*s1, color='blue', marker='.' , label='training set')
        plt.scatter(m2 + self.test_pred*s2, m2 + self.dp.Y_test*s2, color='red', marker='.' , label='test set')
        plt.legend(loc=2, fontsize=14)
        plt.xlabel('pIC50 Pred')  # x軸のラベル
        plt.ylabel('pIC50 Expt')  # y軸のラベル
        #plt.xlim(2.5, 7.5)
        #plt.ylim(2.5, 7.5)
        text1 = f'r2_train={self.r2_train}'
        text2 = f'r2_test={self.r2_test}'
        plt.text(4.9, 3, text1, fontsize="xx-large", color='blue')
        plt.text(4.9, 2.5, text2, fontsize="xx-large", color='red')
        plt.savefig(f"{self.directory}/yy_plot_{self.basename}_"+str(i)+".png")

    def save_shap(self, i):
        # SHAP値を計算するためのexplainerを作成
        explainer = shap.TreeExplainer(
            model = self.best_model,  # 学習済みモデル
            feature_perturbation="interventional"  # 推奨
            )
        # SHAP値を計算
        shap_values = explainer(self.dp.X_train, self.dp.Y_train)

        # 棒グラフで重要度を可視化
        fig2 = plt.figure()   #新しいウィンドウを描画
        fig2.subplots()
        shap.plots.bar(shap_values=shap_values, max_display=10, show=False)
        fig2.savefig(f"{self.directory}/shap_bar_{self.basename}_"+str(i)+".png") # 画像を保存
        
        fig3 = plt.figure()   #新しいウィンドウを描画
        fig3.subplots()
        shap.plots.beeswarm(shap_values=shap_values, max_display=10, show=False)
        fig3.savefig(f"{self.directory}/shap_beeswarm_{self.basename}_"+str(i)+".png", bbox_inches='tight') 
        plt.clf()
        plt.close()


In [7]:
def main():
    all_data = pd.read_csv('231202_tateishi_a-glucosidase_ECFP4.csv')
    data = all_data.drop(['comp_name', 'ChEMBL ID', 'Smiles', 'IC50(nM)', 'IC50(M)', 'pIC50'], axis=1)
    desc_set = data.columns.values
    #print(desc_set)
    #print(len(desc_set))
    object_variable = "pIC50"
    directory = "model_result"
    basename = "ECFP4"
    train_score = []
    test_score =[]
    bayes_n_trials = 1
    n_seed = 1
    seed_list = pd.read_csv("../datasets/split_seed_result.csv")["seed"]
    if not os.path.isdir(f"{directory}"):
        os.makedirs(f"{directory}")

    #ここで全データでベイズ最適化
    all_data_processor = DataProcessor(all_data, desc_set, object_variable, split_seed=0)
    opt = Optimization(all_data_processor, directory, basename, n_trials=bayes_n_trials)
    opt.optimize_parameters()
    
    for i, split_seed in enumerate(seed_list):
        data_processor = DataProcessor(all_data, desc_set, object_variable, split_seed=split_seed)
        modeler = Modeling(data_processor,opt, directory, basename, n_seed=n_seed)
        modeler.search_best_model(i)

        saver = ModelSaver(modeler, data_processor, directory, basename)
        _, _, r2_train, r2_test = saver.save_best_model(i)
        train_score.append(r2_train)
        test_score.append(r2_test)
        
        # 図の保存
        #saver.save_yyplot(i)
        #saver.save_shap(i)

    train_score_df = pd.DataFrame(train_score)
    test_score_df = pd.DataFrame(test_score)
    score_df = pd.concat([train_score_df, test_score_df], axis=1)
    score_df.columns = ['r2_train','r2_test']
    score_df.to_csv(f'{directory}/all_score.csv')
    
if __name__ == "__main__":
    main()

[I 2023-12-06 10:00:00,167] A new study created in memory with name: no-name-df5895fb-765d-4703-a100-229861c7c878
enqueue_trial is experimental (supported from v1.2.0). The interface can change in the future.
create_trial is experimental (supported from v2.0.0). The interface can change in the future.
add_trial is experimental (supported from v2.0.0). The interface can change in the future.
[I 2023-12-06 10:00:22,458] Trial 0 finished with value: 0.4269018748789085 and parameters: {'max_depth': 5, 'max_leaves': 4, 'colsample_bytree': 0.3402744077431494, 'colsample_bylevel': 0.36271273740495225, 'colsample_bynode': 0.7347853808112024, 'subsample': 0.8515695846938984, 'reg_lambda': 0.0003194232244540776, 'reg_alpha': 0.41955205249976124, 'leraning_rate': 8.078050956559998, 'gamma': 0.13872463795888207}. Best is trial 0 with value: 0.4269018748789085.


Spliting Data 0 : seed 0 done, best_seed = 0, best_score = 0.7663
Spliting Data 1 : seed 0 done, best_seed = 0, best_score = 0.7719
Spliting Data 2 : seed 0 done, best_seed = 0, best_score = 0.6937
Spliting Data 3 : seed 0 done, best_seed = 0, best_score = 0.8471
Spliting Data 4 : seed 0 done, best_seed = 0, best_score = 0.8188
Spliting Data 5 : seed 0 done, best_seed = 0, best_score = 0.8083
Spliting Data 6 : seed 0 done, best_seed = 0, best_score = 0.8553
Spliting Data 7 : seed 0 done, best_seed = 0, best_score = 0.8487
Spliting Data 8 : seed 0 done, best_seed = 0, best_score = 0.8714
Spliting Data 9 : seed 0 done, best_seed = 0, best_score = 0.8184
Spliting Data 10 : seed 0 done, best_seed = 0, best_score = 0.7826
Spliting Data 11 : seed 0 done, best_seed = 0, best_score = 0.7477
Spliting Data 12 : seed 0 done, best_seed = 0, best_score = 0.7355
Spliting Data 13 : seed 0 done, best_seed = 0, best_score = 0.7999


KeyboardInterrupt: 